# IDM Project - Real-Time Object Detection with YOLOv8
Course: CSET225 | Intelligent Design Models | Summer Term
Student: Sanskar Singh

This notebook runs the full pipeline on Google Colab (T4 GPU). For local execution use src/train.py instead.

## 1. Environment Setup
Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install -q ultralytics
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Load Pretrained YOLOv8 and Run Baseline Inference
Load yolov8n.pt and run on a sample image to verify the pipeline works before fine-tuning.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

model = YOLO('yolov8n.pt')
# Run on a sample bus image from ultralytics assets
results = model.predict('https://ultralytics.com/images/bus.jpg', conf=0.25, save=True)
display(Image.open(Path(results[0].save_dir) / 'bus.jpg'))

## 3. Fine-tune on COCO128
COCO128 auto-downloads. Swap the data yaml for your own dataset (Roboflow download) to adapt to a target domain.

In [ ]:
# Reload FRESH pretrained model (don't reuse already-trained model object)
model = YOLO('yolov8n.pt')

# Fine-tune with config appropriate for a tiny (128-image) dataset
# - lr0=0.001: low LR preserves pretrained features
# - freeze=10: freeze backbone (layers 0-9), train head only
# - epochs=15: fewer epochs to avoid overfitting on 128 images
results = model.train(
    data='coco128.yaml',
    epochs=15,
    imgsz=640,
    batch=16,
    device=0,
    lr0=0.001,
    freeze=10,
    patience=20,
    name='train_fresh'
)


## 4. Evaluate
Print mAP50, mAP50-95, precision, recall from the validation set.

In [ ]:
from pathlib import Path
import os, glob

# Find best.pt (Ultralytics may nest the save dir)
matches = glob.glob('**/best.pt', recursive=True)
weights_path = matches[0] if matches else 'runs/detect/train/weights/best.pt'
print(f"Loading weights: {weights_path}")

best = YOLO(weights_path)
metrics = best.val(data='coco128.yaml')
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")


## 5. Inference on Sample Images

Run detection on sample images without any upload required. Displays annotated results inline.

In [ ]:
from pathlib import Path
import cv2, glob, os
from ultralytics import YOLO
from PIL import Image

# Load the fine-tuned model (mAP50=0.72 after correct training config)
matches = sorted(glob.glob('**/best.pt', recursive=True), key=os.path.getmtime, reverse=True)
weights_path = matches[0] if matches else 'runs/detect/train_fresh/weights/best.pt'
model_ft = YOLO(weights_path)
print(f"Using fine-tuned weights: {weights_path}")

# Run inference on sample images
for url in ['https://ultralytics.com/images/bus.jpg',
            'https://ultralytics.com/images/zidane.jpg']:
    name = url.split('/')[-1]
    res = model_ft.predict(url, conf=0.25)[0]
    annotated = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    display(Image.fromarray(annotated))
    names = [res.names[int(b.cls)] for b in res.boxes]
    print(f"  {name}: {len(res.boxes)} detections — {names}\n")


## 6. Save Trained Weights

Copy the fine-tuned weights to /content/ for easy download from the Colab file browser.

In [ ]:
import glob, shutil, os
from pathlib import Path

# Find newest best.pt (exclude previous copies in /content/)
matches = sorted(
    [m for m in glob.glob('**/best.pt', recursive=True) if os.path.abspath(m) != '/content/best.pt'],
    key=os.path.getmtime, reverse=True
)
weights_path = matches[0] if matches else 'runs/detect/train/weights/best.pt'

dest = '/content/best.pt'
if os.path.abspath(weights_path) != os.path.abspath(dest):
    shutil.copy2(weights_path, dest)
size_mb = Path(dest).stat().st_size / 1024 / 1024
print(f"Weights saved to: {dest}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nDownload via Colab file browser:")
print(f"  Left panel > Files > best.pt > right-click > Download")
